# 4. Field Cancerisation Investigation

This notebook investigates whether stronger architectures can detect **field cancerisation effects** - subtle changes in morphologically normal tissue adjacent to tumors.

## Background: The Field Cancerisation Hypothesis

Field cancerisation (also called "field effect" or "field defect") is a biological phenomenon where tissue that appears histologically normal has already acquired pre-malignant molecular changes due to proximity to a tumor. This could manifest as:

- **Altered cell morphology**: Subtle changes in cell shape, nuclear size, or chromatin patterns
- **Microenvironment changes**: Different extracellular matrix composition, immune infiltration patterns
- **Vascular differences**: Changes in blood vessel density or architecture

## Previous Results

In notebook 03, our `subtle` model achieved:
- **Validation AUC: 0.637** - suggesting it learned *something* from the training data
- **Test AUC: 0.543** - near random chance, suggesting overfitting to spurious correlations

This gap between validation and test performance raises key questions:
1. Is there a genuine field cancerisation signal that stronger models could capture?
2. Are we overfitting to batch effects, staining variations, or slide-level artifacts?

## Approaches Tested

| Approach | Model | Description |
|----------|-------|-------------|
| 1 | subtle | Custom CNN with small kernels (baseline from notebook 03) |
| 2 | attention | Custom CNN + spatial attention mechanism |
| 3 | transfer | Frozen MobileNetV2 feature extractor |
| 4 | transfer_finetune | MobileNetV2 with last 30 layers unfrozen |

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to project directory
%cd /content/drive/MyDrive/new_work/Projects/Camelyon16/camelyon16-pathology

# Install requirements
!pip -q install -r requirements.txt
!apt-get -y install openslide-tools

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import gc

from config import DEFAULT_CONFIG
from src.models import run_binary_experiment, MODEL_REGISTRY, evaluate_on_test_set, load_model_metadata

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Dataset paths
TRAIN_DATASET_PATH = '/content/drive/MyDrive/new_work/Projects/pathovis_project/data/camelyon16_4class_stain_normalised'
TEST_DATASET_PATH = '/content/drive/MyDrive/new_work/Projects/pathovis_project/data/camelyon16_test_stain_normalised'

print("Available models:")
for name in MODEL_REGISTRY.keys():
    print(f"  - {name}")

## Approach 1: Attention Model

The attention model adds a spatial attention mechanism that learns to focus on the most discriminative regions of each patch. This could help if the field cancerisation signal is localised to specific areas rather than distributed across the whole patch.

In [ ]:
# Configure training
DEFAULT_CONFIG.training.max_patches_per_chunk = 400
DEFAULT_CONFIG.training.cycle_length = 2
DEFAULT_CONFIG.training.normalise_patches = True

# Run attention model on experiment 3 (field cancerisation)
print("=" * 60)
print("APPROACH 1: Attention Model")
print("=" * 60)

results_attention = run_binary_experiment(
    dataset_path=TRAIN_DATASET_PATH,
    experiment_type=3,  # normal_from_normal vs normal_from_tumor
    model_name='attention',
    epochs=15,
    learning_rate=1e-5
)

print(f"\nAttention Model Results:")
print(f"  Accuracy: {results_attention['results']['accuracy']:.1%}")
print(f"  AUC: {results_attention['results']['auc']:.3f}")

# Cleanup
tf.keras.backend.clear_session()
gc.collect()

## Approach 2: Frozen Transfer Learning

MobileNetV2 pretrained on ImageNet provides rich, generic image features. By freezing the base model, we test whether these off-the-shelf features can distinguish normal tissue from normal vs tumour slides.

Advantages:
- Learned from 1M+ images - much more diverse than our dataset
- Features capture textures, edges, and patterns at multiple scales
- Fast training since only the classification head updates

In [ ]:
# Run frozen transfer model
print("=" * 60)
print("APPROACH 2: Frozen Transfer Learning (MobileNetV2)")
print("=" * 60)

results_transfer = run_binary_experiment(
    dataset_path=TRAIN_DATASET_PATH,
    experiment_type=3,
    model_name='transfer',
    epochs=15,
    learning_rate=1e-5
)

print(f"\nFrozen Transfer Results:")
print(f"  Accuracy: {results_transfer['results']['accuracy']:.1%}")
print(f"  AUC: {results_transfer['results']['auc']:.3f}")

# Cleanup
tf.keras.backend.clear_session()
gc.collect()

## Approach 3: Fine-tuned Transfer Learning

Fine-tuning unfreezes the last 30 layers of MobileNetV2, allowing them to adapt to histopathology images while keeping lower-level features (edges, textures) frozen.

This is a middle ground:
- More capacity to learn domain-specific features than frozen transfer
- More regularisation than training from scratch (early layers constrained)

In [ ]:
# Run fine-tuned transfer model
print("=" * 60)
print("APPROACH 3: Fine-tuned Transfer Learning (MobileNetV2)")
print("=" * 60)

results_finetune = run_binary_experiment(
    dataset_path=TRAIN_DATASET_PATH,
    experiment_type=3,
    model_name='transfer_finetune',
    epochs=15,
    learning_rate=1e-5
)

print(f"\nFine-tuned Transfer Results:")
print(f"  Accuracy: {results_finetune['results']['accuracy']:.1%}")
print(f"  AUC: {results_finetune['results']['auc']:.3f}")

# Cleanup
tf.keras.backend.clear_session()
gc.collect()

## Comparison: Validation AUC Across All Approaches

Load the subtle model results from notebook 03 and compare all four approaches.

In [ ]:
# Load subtle model metadata from notebook 03
try:
    subtle_meta = load_model_metadata('./models/slide_context_detection.keras')
    subtle_val_auc = subtle_meta['auc']
    print(f"Loaded subtle model: val AUC = {subtle_val_auc:.3f}")
except FileNotFoundError:
    print("Subtle model not found - using placeholder value from notebook 03")
    subtle_val_auc = 0.637  # From previous run

# Collect all results
approaches = ['Subtle\n(baseline)', 'Attention', 'Transfer\n(frozen)', 'Transfer\n(fine-tuned)']
val_aucs = [
    subtle_val_auc,
    results_attention['results']['auc'],
    results_transfer['results']['auc'],
    results_finetune['results']['auc']
]

# Create comparison bar chart
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['steelblue', 'darkorange', 'forestgreen', 'crimson']
bars = ax.bar(approaches, val_aucs, color=colors, edgecolor='black', linewidth=1.2)

# Add value labels on bars
for bar, auc in zip(bars, val_aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{auc:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Reference lines
ax.axhline(0.5, color='gray', linestyle='--', linewidth=2, label='Random chance')
ax.axhline(subtle_val_auc, color='steelblue', linestyle=':', alpha=0.7, label=f'Subtle baseline ({subtle_val_auc:.3f})')

ax.set_ylabel('Validation AUC', fontsize=12)
ax.set_title('Field Cancerisation Detection: Validation AUC Comparison\n(Experiment 3: normal_from_normal vs normal_from_tumor)', fontsize=12)
ax.set_ylim(0.4, 0.9)
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('field_cancerisation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary table
print("\n" + "=" * 50)
print("SUMMARY: Validation AUC")
print("=" * 50)
for approach, auc in zip(approaches, val_aucs):
    approach_clean = approach.replace('\n', ' ')
    print(f"{approach_clean:25s}: {auc:.3f}")

## Test Set Evaluation

Evaluate the best-performing model on the held-out test set. This is the critical check for whether the model learned genuine features or overfit to training artifacts.

In [ ]:
# Find the best model based on validation AUC
model_results = {
    'attention': results_attention,
    'transfer': results_transfer,
    'transfer_finetune': results_finetune
}

best_model_name = max(model_results, key=lambda k: model_results[k]['results']['auc'])
best_results = model_results[best_model_name]

print(f"Best model: {best_model_name} (val AUC = {best_results['results']['auc']:.3f})")

# Get normalise setting from training config
normalise_setting = DEFAULT_CONFIG.training.normalise_patches
print(f"Normalise patches: {normalise_setting}")

# Evaluate on test set
print("\n" + "=" * 60)
print(f"TEST SET EVALUATION: {best_model_name}")
print("=" * 60)

exp3_class_mapping = {0: ['normal_from_normal'], 1: ['normal_from_tumor']}
test_results = evaluate_on_test_set(
    best_results['model'],
    TEST_DATASET_PATH,
    exp3_class_mapping,
    f"exp3_{best_model_name}",
    threshold=best_results['results']['threshold'],
    normalise=normalise_setting
)

print(f"\nValidation AUC: {best_results['results']['auc']:.3f}")
print(f"Test AUC: {test_results['auc']:.3f}")
print(f"Gap: {best_results['results']['auc'] - test_results['auc']:.3f}")
print(f"\nTest Accuracy: {test_results['accuracy']:.1%}")
print(test_results['report'])

## Conclusion

*Placeholder for interpreting results after running the experiments.*

### Key Questions to Address:

1. **Did any architecture significantly outperform the subtle baseline on validation?**
   - If yes: suggests potential for detecting field cancerisation with the right architecture
   - If no: the limitation may be in the data itself

2. **How large is the validation-to-test gap for each approach?**
   - Large gap (>0.1 AUC): suggests overfitting to slide-level artifacts
   - Small gap (<0.05 AUC): suggests genuine generalisable signal

3. **Does transfer learning help?**
   - Frozen transfer: tests if ImageNet features are relevant for this task
   - Fine-tuned transfer: tests if domain adaptation improves detection

### Possible Outcomes:

| Scenario | Val AUC | Test AUC | Interpretation |
|----------|---------|----------|----------------|
| A | High (>0.7) | High (>0.65) | Field cancerisation signal exists and generalises |
| B | High (>0.7) | Low (~0.5) | Overfitting to slide artifacts, not tissue biology |
| C | Low (~0.55) | Low (~0.5) | No detectable signal at patch level |

### Next Steps (if overfitting persists):

- **Slide-aware splitting**: Ensure no patient appears in both train and test
- **Stronger regularisation**: More dropout, weight decay, smaller models
- **Multi-instance learning**: Aggregate predictions across multiple patches per slide
- **Stain normalisation**: Apply Macenko or Reinhard normalisation